#  Introducción ML · De explicar a predecir

**Analítica de datos aplicada a la Ingeniería Civil** 

**Caso:** resistencia a compresión del hormigón (1030 mezclas ensayadas · dataset de Yeh, UCI Machine Learning Repository)

> 🎯 **Mensaje clave:** un modelo vale lo que acierta con datos que no ha visto.

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Cargar los datos del hormigón |
| 2 | Reproducir la regresión múltiple con scikit-learn |
| 3 | Separar entrenamiento y prueba: ¿se mantiene el R²? |
| 4 | Polinomios: de la edad a la curva en U |
| 5 | KNN variando K |
| 6 | Tabla de modelos y conclusiones |

### Imports y configuración básica

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Colores que usaremos en todo el bloque
AZUL = "#4C72B0"      # datos de entrenamiento
NARANJA = "#DD8452"   # datos de prueba
ROJO = "#C44E52"      # modelo

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)

---
## 1. Cargar los datos del hormigón

Mismo fichero y misma ruta que en los notebooks de regresión que visteis en las sesiones anteriores

In [ ]:
ruta = "/workspaces/AnaliticaDeDatos/datos/clases/clase_1_analisis_exploratorio_de_datos/Concrete_Data.xls"
concrete_data = pd.read_excel(ruta, index_col=False)

# Los nombres originales son muy largos: los cambiamos por nombres cortos en español
# (mismo orden que las columnas del fichero original)
concrete_data.columns = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
                         "arido_grueso", "arido_fino", "edad", "resistencia"]

print("Dimensiones del dataset:", concrete_data.shape)
concrete_data.head()

| Variable | Significado | Unidades |
|---|---|---|
| `cemento`, `escoria`, `cenizas`, `agua`, `superplastificante`, `arido_grueso`, `arido_fino` | Dosificación de la mezcla | kg por m³ de hormigón |
| `edad` | Días desde el hormigonado hasta el ensayo | días |
| `resistencia` | Resistencia a compresión medida (**variable respuesta**) | MPa |

In [ ]:
# ¿A qué edades se ensayaron las probetas?
concrete_data["edad"].value_counts().sort_index()

In [ ]:
# Separamos las variables de entrada (X) y la respuesta (y)
variables = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
             "arido_grueso", "arido_fino", "edad"]

X = concrete_data[variables]
y = concrete_data["resistencia"]

> ### 💡 Preguntas
> 
> 1) ¿Cuántas mezclas y cuántas variables de entrada tenemos?
> 
> 2) La edad, ¿toma cualquier valor o solo unos pocos? ¿Por qué creéis que es así?

---
## 2. Reproducir la regresión múltiple con scikit-learn

En la sesión anterior ajustasteis la regresión múltiple con **mínimos cuadrados** usando `np.linalg.lstsq`. Repetimos ese cálculo y lo comparamos con **scikit-learn**, la librería que usaremos en todo el bloque.

Con scikit-learn **todos los modelos** siguen el mismo patrón: crear → `fit` (entrenar) → `predict` (predecir) → `score` (evaluar).

### Formulación

**Regresión lineal múltiple.** Con $p$ variables de entrada (aquí $p = 8$) y $n$ mezclas:

$$y_i = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip} + \varepsilon_i \qquad \Longleftrightarrow \qquad \mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

donde $\mathbf{X}$ es la matriz de diseño de dimensión $n \times (p+1)$ (con una primera columna de unos para el intercepto $\beta_0$).

**Mínimos cuadrados.** Se eligen los coeficientes que minimizan la suma de los cuadrados de los residuos (SCE):

$$\hat{\boldsymbol{\beta}} = \arg\min_{\boldsymbol{\beta}} \; \text{SCE} = \arg\min_{\boldsymbol{\beta}} \sum_{i=1}^{n} \left(y_i - \hat y_i\right)^2 = \arg\min_{\boldsymbol{\beta}} \left\lVert \mathbf{y} - \mathbf{X}\boldsymbol{\beta} \right\rVert^2$$

con solución $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}$.

**Descomposición de la variabilidad.**

$$\underbrace{\sum_{i=1}^{n} (y_i - \bar y)^2}_{\text{SCT (total)}} \;=\; \underbrace{\sum_{i=1}^{n} (\hat y_i - \bar y)^2}_{\text{SCR (regresión, explicada)}} \;+\; \underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{SCE (error, residual)}}$$

**Coeficiente de determinación.**

$$R^2 = \frac{\text{SCR}}{\text{SCT}} = 1 - \frac{\text{SCE}}{\text{SCT}}$$

Esta descomposición (y la igualdad entre las dos formas de $R^2$) es válida para el ajuste por mínimos cuadrados con intercepto sobre los datos de entrenamiento. En datos de prueba, `score` calcula $R^2 = 1 - \text{SCE}/\text{SCT}$, que puede ser negativo.

In [ ]:
# Lo que ya hicimos en la sesión anterior: mínimos cuadrados "a mano"
X_diseno = np.column_stack([np.ones(len(X)), X.values])   # columna de 1s para el intercepto
beta, _, _, _ = np.linalg.lstsq(X_diseno, y.values, rcond=None)

✏️ **Rellena el hueco:** ajusta el modelo con los **mismos datos** que hemos usado con `lstsq`.

In [ ]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(___, ___)   # ✏️ ¿con qué datos lo ajustamos?

# Comparamos los coeficientes de los dos métodos
coef_sklearn = np.r_[modelo_lineal.intercept_, modelo_lineal.coef_]
comparacion = pd.DataFrame({"lstsq": beta, "scikit-learn": coef_sklearn},
                           index=["intercepto"] + variables)
display(comparacion.round(4))

print("¿Coinciden los coeficientes?", np.allclose(beta, coef_sklearn))
print(f"R² calculado con los mismos datos del ajuste: {modelo_lineal.score(X, y):.3f}")

> ### 💡 Preguntas
> 
> 1) ¿Coinciden los coeficientes? ¿Por qué?
> 
> 2) ¿Qué signo tiene el coeficiente del agua? ¿Tiene sentido para un ingeniero?
> 
> 3) Este R², ¿con qué datos se ha calculado? ¿Nos dice cómo funcionará el modelo con una mezcla nueva?

---
## 3. Separar entrenamiento y prueba

Apartamos un 20 % de las mezclas como **prueba**: el modelo no las verá al entrenar. Así simulamos «las mezclas nuevas que llegan mañana de la planta».

✏️ **Rellena el hueco:** queremos un 20 % de mezclas para prueba.

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=___, random_state=42)   # ✏️

print("Entrenamiento:", len(X_ent), "mezclas")
print("Prueba:       ", len(X_pru), "mezclas")

✏️ **Rellena el hueco:** calcula el R² en el conjunto de **prueba**.

In [ ]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_ent, y_ent)          # ¡solo con las mezclas de entrenamiento!

r2_ent_lineal = modelo_lineal.score(X_ent, y_ent)
r2_pru_lineal = modelo_lineal.score(___, ___)   # ✏️

print(f"R² entrenamiento: {r2_ent_lineal:.3f}")
print(f"R² prueba:        {r2_pru_lineal:.3f}")

In [ ]:
def grafica_obs_pred(modelo, titulo):
    """Observado frente a predicho, en entrenamiento y en prueba."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
    for ax, Xs, ys, color, nombre in [(axes[0], X_ent, y_ent, AZUL, "Entrenamiento"),
                                      (axes[1], X_pru, y_pru, NARANJA, "Prueba")]:
        y_pred = modelo.predict(Xs)
        ax.scatter(ys, y_pred, color=color, alpha=0.7, edgecolor="white", linewidth=0.5, s=40, zorder=2)
        ax.plot([0, 85], [0, 85], color=ROJO, linewidth=2, linestyle="--", zorder=1, label="Ajuste perfecto")
        ax.set_title(f"{nombre} · R² = {r2_score(ys, y_pred):.3f}")
        ax.set_xlabel("Resistencia observada (MPa)")
        ax.set_aspect("equal", adjustable="box")
        estilo(ax)
    axes[0].set_ylabel("Resistencia predicha (MPa)")
    axes[0].legend(loc="upper left", frameon=False)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


grafica_obs_pred(modelo_lineal, "Regresión lineal múltiple (8 variables)")

**¿Y si el reparto hubiera sido otro?** `random_state` fija el azar del reparto. Probamos varias semillas:

In [ ]:
for semilla in [0, 1, 2, 3, 4, 42]:
    Xe, Xp, ye, yp = train_test_split(X, y, test_size=0.2, random_state=semilla)
    m = LinearRegression().fit(Xe, ye)
    print(f"semilla = {semilla:2d} → R² entrenamiento = {m.score(Xe, ye):.3f} | R² prueba = {m.score(Xp, yp):.3f}")

Empezamos la **tabla de modelos de la resistencia del hormigón**. Cada modelo nuevo se anotará aquí, evaluado de la misma forma.

In [ ]:
tabla_modelos = []


def anotar(modelo, r2_ent, r2_pru, comentario=""):
    """Añade una fila a la tabla de modelos del hormigón."""
    tabla_modelos.append({"Modelo": modelo,
                          "R² entrenamiento": round(r2_ent, 3),
                          "R² prueba": round(r2_pru, 3),
                          "Comentario": comentario})


anotar("Regresión lineal", r2_ent_lineal, r2_pru_lineal, "8 variables")
pd.DataFrame(tabla_modelos)

> ### 💡 Preguntas
> 
> 1) ¿Se mantiene el R² con mezclas que el modelo no ha visto? ¿Esperabais una caída mayor?
> 
> 2) ¿Por qué cambia el R² de prueba al cambiar la semilla? ¿Qué problema supone?
> 
> 3) ¿Qué creéis que pasaría con un modelo mucho más flexible que una regresión lineal?

---
## 4. Polinomios: la curva en U

La resistencia **no crece en línea recta** con la edad: crece rápido los primeros días y luego se estabiliza. Vamos a probar curvas cada vez más flexibles: **polinomios de grado creciente**.

### 4.1. Ver las curvas: polinomios sobre la edad

Primero usamos solo la edad, para poder dibujar las curvas, y **solo unas pocas mezclas de entrenamiento** (`n_pequeno`), para que el efecto se vea con claridad.

In [ ]:
n_pequeno = 40   # nº de mezclas de entrenamiento que usamos en este apartado

rng = np.random.default_rng(0)
idx = rng.choice(len(X_ent), size=n_pequeno, replace=False)

x_ent_edad = X_ent[["edad"]].iloc[idx]
y_ent_edad = y_ent.iloc[idx]
x_pru_edad = X_pru[["edad"]]                      # la prueba es la misma de antes

edad_malla = pd.DataFrame({"edad": np.linspace(1, 365, 500)})   # para dibujar las curvas


def modelo_polinomico(grado):
    """Regresión con un polinomio de grado `grado` de las variables de entrada.
    Escalamos antes de elevar a potencias: si no, edad^10 ≈ 10^25 y los cálculos fallan
    (el escalado lo veremos con calma en la siguiente sesión)."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(grado), LinearRegression())

✏️ **Rellena el hueco:** elige un grado **alto** para el tercer panel (por ejemplo, 10).

In [ ]:
grados_ejemplo = [1, 3, ___]   # ✏️

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, g in zip(axes, grados_ejemplo):
    modelo = modelo_polinomico(g).fit(x_ent_edad, y_ent_edad)
    r2_e = modelo.score(x_ent_edad, y_ent_edad)
    r2_p = modelo.score(x_pru_edad, y_pru)

    ax.scatter(x_pru_edad, y_pru, color=NARANJA, alpha=0.35, s=20, label="Prueba", zorder=1)
    ax.scatter(x_ent_edad, y_ent_edad, color=AZUL, edgecolor="white", s=45, label="Entrenamiento", zorder=3)
    ax.plot(edad_malla, modelo.predict(edad_malla), color=ROJO, linewidth=2.2, zorder=4)

    ax.set_ylim(-5, 95)                 # recortamos el eje: las curvas muy flexibles se disparan
    ax.set_title(f"Grado {g}\nR² entren. = {r2_e:.2f} | R² prueba = {r2_p:.2f}")
    ax.set_xlabel("Edad (días)")
    estilo(ax)
axes[0].set_ylabel("Resistencia (MPa)")
axes[0].legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

> ### 💡 Preguntas
> 
> 1) ¿Qué grado se queda corto (infraajuste)? ¿Cuál se pasa (sobreajuste)?
> 
> 2) En el grado alto, ¿qué ocurre entre las edades ensayadas y en los extremos?

> ### 💡 Experimento
> 
> Vuelve a la celda de `n_pequeno`, cámbialo a `200` y ejecuta de nuevo las dos celdas. ¿Se calman las curvas? ¿Por qué?

### 4.2. La curva en U con las 8 variables

La edad sola explica poco de la resistencia (mezclas de la misma edad tienen dosificaciones muy distintas). Para medir el sobreajuste de verdad usamos **las 8 variables** y polinomios de grado creciente, que incluyen también productos entre variables (por ejemplo, `cemento × edad`).

- **Grado 1** = la regresión lineal de antes (9 parámetros).
- Al subir el grado, el número de parámetros se dispara. Con grado 4 hay casi 500 parámetros… para 824 mezclas de entrenamiento.


**Recuento de parámetros polinómicos**

`PolynomialFeatures(grado)` genera todos los monomios de grado menor o igual que el grado elegido, incluidas las interacciones entre variables. Cada monomio aporta una columna y un coeficiente.

Con 8 variables y grado 2 tenemos:

- Intercepto: 1 parámetro.
- Términos lineales: 8 parámetros, uno por variable.
- Cuadrados: 8 parámetros, uno por variable.
- Productos cruzados: $\binom{8}{2} = 28$ parámetros.

En total: **1 + 8 + 8 + 28 = 45 parámetros**.

Para $p$ variables y grado máximo $d$, contando el intercepto:

$$N_{\mathrm{par}} = \binom{p+d}{d} = \frac{(p+d)!}{p!\,d!}$$

Con 8 variables, el recuento es:

- Grado 1: 9 parámetros.
- Grado 2: 45 parámetros.
- Grado 3: 165 parámetros.
- Grado 4: 495 parámetros.
- Grado 5: 1287 parámetros.

En scikit-learn, el número de columnas polinómicas se consulta con `n_output_features_`.



✏️ **Rellena el hueco:** queremos probar los grados del 1 al 5.

In [ ]:
grados = list(range(1, ___))   # ✏️ range(1, N) va de 1 a N-1

n_param, mse_ent, mse_pru, r2_ent_g, r2_pru_g = [], [], [], [], []
for g in grados:
    modelo = modelo_polinomico(g).fit(X_ent, y_ent)
    n_param.append(modelo.named_steps["polynomialfeatures"].n_output_features_)
    mse_ent.append(mean_squared_error(y_ent, modelo.predict(X_ent)))
    mse_pru.append(mean_squared_error(y_pru, modelo.predict(X_pru)))
    r2_ent_g.append(modelo.score(X_ent, y_ent))
    r2_pru_g.append(modelo.score(X_pru, y_pru))

display(pd.DataFrame({"Grado": grados, "Nº de parámetros": n_param,
                      "R² entrenamiento": np.round(r2_ent_g, 3), "R² prueba": np.round(r2_pru_g, 3)}))

In [ ]:
grado_mejor = grados[int(np.argmin(mse_pru))]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grados, mse_ent, marker="o", color=AZUL, linewidth=2, label="Entrenamiento")
ax.plot(grados, mse_pru, marker="o", color=NARANJA, linewidth=2, label="Prueba")
ax.axvline(grado_mejor, color="grey", linestyle=":", label=f"Mínimo en prueba (grado {grado_mejor})")
ax.set_ylim(0, 3 * min(mse_pru))       # recortamos el eje: con grado alto el error de prueba se dispara
ax.set_xticks(grados)
ax.set_xlabel("Grado del polinomio (flexibilidad →)")
ax.set_ylabel("Error cuadrático medio (MPa²)")
ax.set_title("Curva de error: polinomios con las 8 variables")
ax.legend(frameon=False, loc="lower left")
estilo(ax)
plt.show()

> ### 💡 Preguntas
> 
> 1) ¿Qué le pasa al error de entrenamiento al subir el grado? ¿Y al de prueba?
> 
> 2) ¿Qué ocurre cuando el número de parámetros se acerca al número de mezclas de entrenamiento?
> 
> 3) ¿Dónde está el punto óptimo? ¿Con qué datos lo hemos elegido? (Recordad la «pregunta trampa» de la presentación)

In [ ]:
# Anotamos en la tabla el polinomio del mejor grado
anotar(f"Polinomio (grado {grado_mejor})", r2_ent_g[grado_mejor - 1], r2_pru_g[grado_mejor - 1],
       "8 variables · grado elegido mirando la prueba")
pd.DataFrame(tabla_modelos)

---
## 5. KNN variando K

**KNN** predice la resistencia de una mezcla como la **media de las K mezclas más parecidas** del entrenamiento. No hay ecuación: el modelo son los propios datos. **K es el mando de la complejidad.**

### 5.1. KNN con la edad: ver el efecto de K

✏️ **Rellena el hueco:** el número de vecinos de cada panel es `k`.

In [ ]:
ks = [1, 20, 400]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, k in zip(axes, ks):
    knn = KNeighborsRegressor(n_neighbors=___)   # ✏️
    knn.fit(X_ent[["edad"]], y_ent)
    r2_p = knn.score(X_pru[["edad"]], y_pru)

    ax.scatter(X_ent["edad"], y_ent, color=AZUL, alpha=0.35, s=20, label="Entrenamiento", zorder=1)
    ax.plot(edad_malla, knn.predict(edad_malla), color=ROJO, linewidth=2.2, drawstyle="steps-mid",
            label="Predicción KNN", zorder=3)
    ax.set_title(f"K = {k}\nR² prueba = {r2_p:.2f}")
    ax.set_xlabel("Edad (días)")
    estilo(ax)
axes[0].set_ylabel("Resistencia (MPa)")
axes[0].legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

> ### 💡 Preguntas
> 
> 1) ¿Qué panel sobreajusta? ¿Cuál tiene más sesgo?
> 
> 2) Con K = 400, ¿qué está haciendo en realidad el modelo?

### 5.2. KNN con las 8 variables: elegir K

Ahora usamos **todas las variables**, como la regresión lineal, y recorremos K de 1 a 50.

> ⚠️ De momento **no escalamos** las variables. Como KNN mide distancias, el cemento (cientos de kg) pesa mucho más que el superplastificante (decenas). En la próxima sesión veremos cómo arreglarlo.

In [ ]:
K_values = list(range(1, 51))
r2_ent_knn, r2_pru_knn = [], []

for k in K_values:
    knn = KNeighborsRegressor(n_neighbors=k).fit(X_ent, y_ent)
    r2_ent_knn.append(knn.score(X_ent, y_ent))
    r2_pru_knn.append(knn.score(X_pru, y_pru))

✏️ **Rellena el hueco:** el mejor K es el que da el **mayor R² en prueba**.

In [ ]:
k_mejor = K_values[int(np.argmax(___))]   # ✏️

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_values, r2_ent_knn, marker="o", markersize=4, color=AZUL, linewidth=2, label="KNN · entrenamiento")
ax.plot(K_values, r2_pru_knn, marker="o", markersize=4, color=NARANJA, linewidth=2, label="KNN · prueba")
ax.axhline(r2_pru_lineal, color=ROJO, linestyle="--", label=f"Regresión lineal · prueba ({r2_pru_lineal:.2f})")
ax.axvline(k_mejor, color="grey", linestyle=":", label=f"Mejor K en prueba = {k_mejor}")
ax.set_xlabel("K (número de vecinos)   ← más flexible · más rígido →")
ax.set_ylabel("R²")
ax.set_title("KNN con las 8 variables")
ax.legend(frameon=False, loc="lower left")
estilo(ax)
plt.show()

print(f"Mejor K = {k_mejor} → R² entrenamiento = {r2_ent_knn[k_mejor - 1]:.3f} | R² prueba = {r2_pru_knn[k_mejor - 1]:.3f}")

In [ ]:
knn_mejor = KNeighborsRegressor(n_neighbors=k_mejor).fit(X_ent, y_ent)
grafica_obs_pred(knn_mejor, f"KNN con las 8 variables (K = {k_mejor})")

anotar(f"KNN (K = {k_mejor})", r2_ent_knn[k_mejor - 1], r2_pru_knn[k_mejor - 1],
       "8 variables sin escalar · K elegido mirando la prueba")

> ### 💡 Preguntas
> 
> 1) ¿Cuánto vale el R² de entrenamiento con K = 1? ¿Por qué?
> 
> 2) Esta vez el eje de flexibilidad va al revés que con el polinomio. ¿Dónde está el sobreajuste en esta gráfica?
> 
> 3) ¿Supera KNN a la regresión lineal? ¿Qué nos dice eso sobre la relación entre dosificación y resistencia?
> 
> 4) ¿Con qué datos hemos elegido K? ¿Es fiable el R² de prueba de ese K?

---
## 6. Tabla de modelos y conclusiones

La tabla crecerá en cada sesión, siempre con los mismos datos y la misma forma de evaluar. La guardamos para continuarla más adelante.

In [ ]:
tabla = pd.DataFrame(tabla_modelos)
display(tabla)

tabla.to_csv("tabla_modelos_hormigon.csv", index=False)
print("Tabla guardada en tabla_modelos_hormigon.csv")

> ### 💡 Preguntas finales
> 
> 1) Volvamos a la pregunta del principio: **¿os fiaríais del R² = 0,62?** ¿Qué habéis comprobado?
> 
> 2) ¿Qué modelo llevaríais hoy a la planta? ¿Qué os haría dudar de esa elección?

### 📌 Para llevarse

- **Entrenar ≠ evaluar.** El error de entrenamiento siempre mejora al aumentar la flexibilidad; el que importa es el de **prueba**.
- **Curva en U:** poca flexibilidad → infraajuste (sesgo); demasiada → sobreajuste (varianza).
- **Hiperparámetros** como el grado del polinomio o la K de KNN controlan la flexibilidad. Hoy los hemos elegido mirando la prueba… y eso tiene trampa.
- **Más datos** permiten modelos más flexibles sin sobreajustar tanto.

**Próxima sesión: validar y medir.** Validación cruzada, escalado de variables, errores en MPa (MAE y RMSE) y la maldición de la dimensionalidad.